# Clasificacion de Textos Historicos — Etapa 2: Deep Learning

**Intento 4** — Aprendizaje de Maquina 2026-10, Universidad de los Andes

## Cambios principales respecto al Intento 3

1. **Limpieza mejorada**: mapeo de caracteres historicos (long-s, rotunda-r, ligaduras OCR) detectados por analisis del corpus
2. **Hardware adaptativo**: GPU/CPU con deteccion automatica y override manual; uso de todos los nucleos CPU
3. **MLP Combinado**: dos ramas independientes (word TF-IDF + char TF-IDF) con features artesanales; reemplaza los MLP separados del Intento 3
4. **BiLSTM v3**: multi-head attention en lugar de Bahdanau, mayor capacidad
5. **CharCNN v2**: bloques residuales, mas profundo
6. **LightGBM** (nuevo): gradient boosting sobre TF-IDF combinado, usa todos los nucleos CPU, sin conversion a denso
7. **Eliminados**: Transformer scratch (inconsistente), Jerarquico (complejo sin ganancia estable), CNN v2 (CharCNN es superior), MLP-CharNgram separado (fusionado)

## Orden de ejecucion

| Paso | Descripcion |
|------|-------------|
| 1 | **Obligatorio**: Instalacion, Imports, Hardware+Config |
| 2 | **Obligatorio**: Datos → Limpieza → Split → Augmentacion → Features → Callbacks |
| 3 | Modelos (independientes entre si una vez corrido el paso 2) |
| 4 | Ensemble + Submission (requiere al menos un modelo entrenado) |


In [ ]:
import subprocess, sys
from pathlib import Path


def run(cmd: list) -> bool:
    '''Execute a pip install command, print result, return success flag.'''
    label = ' '.join(cmd[-2:])
    print(f'  -> {label}')
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        tail = result.stderr[-300:] if result.stderr else 'error desconocido'
        print(f'     FALLO: {tail}')
    return result.returncode == 0


print('=' * 65)
print('INSTALADOR DE DEPENDENCIAS — Intento 4')
print('=' * 65)
print(f'Python : {sys.executable}')
print(f'Version: {sys.version.split()[0]}')
print(f'Carpeta: {Path.cwd()}')
print('=' * 65)

run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])

BASE_PACKAGES = [
    'numpy', 'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'tqdm',
]
ML_PACKAGES = [
    'tensorflow==2.15.0',
    'torch',
    'transformers',
    'huggingface-hub',
    'sentencepiece',
    'accelerate',
    'lightgbm',
]

print('\nPaquetes base...')
for pkg in BASE_PACKAGES:
    if not run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg]):
        print(f'     FALLO: {pkg}')

print('\nPaquetes ML/DL...')
for pkg in ML_PACKAGES:
    if not run([sys.executable, '-m', 'pip', 'install', '--upgrade', pkg]):
        print(f'     FALLO: {pkg}')

print('\nInstalacion completa.')


In [ ]:
import os, sys, re, json, random, warnings, unicodedata, pickle, time, multiprocessing
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.sparse import hstack as sparse_hstack

import lightgbm as lgb

import tensorflow as tf
import keras
from keras import layers
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, Callback
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

try:
    import torch
    from torch import nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import (
        AutoTokenizer, AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup,
    )
    BERT_AVAILABLE = True
except ImportError:
    BERT_AVAILABLE = False
    print('PyTorch/Transformers no disponibles. BERT desactivado.')

warnings.filterwarnings('ignore')
print(f'TensorFlow : {tf.__version__} | Keras: {keras.__version__}')
if BERT_AVAILABLE:
    print(f'PyTorch    : {torch.__version__}')
print(f'LightGBM   : {lgb.__version__}')
print(f'NumPy      : {np.__version__} | Pandas: {pd.__version__}')


In [ ]:
# =============================================================================
# CONFIGURACION DE HARDWARE
# DEVICE_MODE: 'auto' usa GPU si existe; 'gpu' fuerza GPU; 'cpu' fuerza CPU
# =============================================================================
DEVICE_MODE = 'auto'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    if DEVICE_MODE == 'cpu':
        tf.config.set_visible_devices([], 'GPU')
        TF_STRATEGY = tf.distribute.OneDeviceStrategy('/CPU:0')
        print(f'TF: Forzando CPU. GPUs disponibles ignoradas: {len(gpus)}')
    elif len(gpus) > 1:
        TF_STRATEGY = tf.distribute.MirroredStrategy()
        print(f'TF: {len(gpus)} GPUs detectadas. Usando MirroredStrategy.')
    else:
        TF_STRATEGY = tf.distribute.OneDeviceStrategy('/GPU:0')
        print(f'TF: GPU detectada -> {gpus[0].name}')
else:
    if DEVICE_MODE == 'gpu':
        raise RuntimeError('DEVICE_MODE="gpu" pero no se detecto ninguna GPU.')
    TF_STRATEGY = tf.distribute.OneDeviceStrategy('/CPU:0')
    NUM_CPU = multiprocessing.cpu_count()
    tf.config.threading.set_intra_op_parallelism_threads(0)
    tf.config.threading.set_inter_op_parallelism_threads(0)
    print(f'TF: Sin GPU. Usando {NUM_CPU} nucleos CPU.')

if BERT_AVAILABLE:
    if DEVICE_MODE == 'cpu':
        DEVICE_PT = torch.device('cpu')
    elif DEVICE_MODE == 'gpu':
        if not torch.cuda.is_available():
            raise RuntimeError('DEVICE_MODE="gpu" pero PyTorch no ve GPU.')
        DEVICE_PT = torch.device('cuda')
    else:
        DEVICE_PT = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'PyTorch dispositivo: {DEVICE_PT}')

# =============================================================================
# FLAGS DE MODELOS — cambiar a False para omitir un modelo
# =============================================================================
TRAIN_MLP     = True
TRAIN_BILSTM  = True
TRAIN_CHARCNN = True
TRAIN_LGBM    = True
TRAIN_BERT    = False   # activar si hay GPU; muy lento en CPU

# =============================================================================
# HIPERPARAMETROS GLOBALES
# =============================================================================
SEED               = 42
MAX_VOCAB          = 80_000
MAX_LEN            = 400
EMBED_DIM          = 200
CHAR_MAX_LEN       = 1_000
CHAR_EMBED_DIM     = 64
BATCH_SIZE         = 32
NUM_EPOCHS         = 100
MIN_TEXT_LEN       = 20
MIN_SAMPLES_CLASS  = 600
LABEL_SMOOTHING    = 0.1
ENSEMBLE_THRESHOLD = 0.05   # val_acc minimo para entrar al ensemble

# =============================================================================
# RUTAS
# =============================================================================
DATA_DIR = Path('./data')
PROCESS  = Path('./process')
for sub in ['keras', 'pkl', 'images', 'bert', 'lgbm']:
    (PROCESS / sub).mkdir(parents=True, exist_ok=True)

kpath = lambda n: str(PROCESS / 'keras' / n)
ppath = lambda n: str(PROCESS / 'pkl'   / n)
ipath = lambda n: str(PROCESS / 'images'/ n)
lpath = lambda n: str(PROCESS / 'lgbm'  / n)

# Semillas de reproducibilidad
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print('Configuracion lista.')
print(f'  MLP={TRAIN_MLP}  BiLSTM={TRAIN_BILSTM}  '
      f'CharCNN={TRAIN_CHARCNN}  LightGBM={TRAIN_LGBM}  BERT={TRAIN_BERT}')


## 1. Carga de datos


In [ ]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_eval  = pd.read_csv(DATA_DIR / 'eval.csv')

print(f'Train : {df_train.shape}')
print(f'Eval  : {df_eval.shape}')
print(f'Columnas train: {df_train.columns.tolist()}')
print(f'Columnas eval : {df_eval.columns.tolist()}')
print(f'Decadas unicas: {sorted(df_train["decade"].unique())}')
print(f'Tipo decade   : {df_train["decade"].dtype}')


## 2. Analisis exploratorio


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Distribucion de decadas
decade_counts = df_train['decade'].value_counts().sort_index()
axes[0].bar(range(len(decade_counts)), decade_counts.values, color='steelblue', alpha=0.8)
axes[0].set_xticks(range(0, len(decade_counts), 5))
axes[0].set_xticklabels(
    [str(d) for i, d in enumerate(decade_counts.index) if i % 5 == 0], rotation=45, fontsize=8)
axes[0].set_title('Distribucion de decadas')
axes[0].set_xlabel('Decada'); axes[0].set_ylabel('Muestras')

# Longitud de textos
lens = df_train['text'].astype(str).str.len()
axes[1].hist(lens, bins=60, color='steelblue', alpha=0.8)
axes[1].axvline(lens.mean(), color='red', linestyle='--', label=f'Media: {lens.mean():.0f}')
axes[1].axvline(lens.median(), color='orange', linestyle='--', label=f'Mediana: {lens.median():.0f}')
axes[1].set_title('Longitud de textos (brutos)')
axes[1].set_xlabel('Chars'); axes[1].legend()

# Longitud media por decada
mean_len_by_decade = df_train.groupby('decade')['text'].apply(lambda x: x.astype(str).str.len().mean())
axes[2].plot(mean_len_by_decade.index, mean_len_by_decade.values, color='steelblue', marker='o', ms=3)
axes[2].set_title('Longitud media por decada')
axes[2].set_xlabel('Decada'); axes[2].set_ylabel('Chars')

plt.tight_layout()
plt.savefig(ipath('eda_intento4.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f'Decadas: {df_train["decade"].nunique()} | Longitud media: {lens.mean():.0f} | Mediana: {lens.median():.0f}')
print(f'Muestras por clase — min: {decade_counts.min()} | max: {decade_counts.max()} | media: {decade_counts.mean():.0f}')


## 3. Limpieza de texto

Mejoras respecto al Intento 3 basadas en analisis del corpus:

- `ſ` (U+017F, long-s): muy frecuente en textos del s.XV-XVIII; se mapea a `s`
- `ꝛ` (U+A75B, rotunda-r): caracter historico; se mapea a `r`
- `¬` (U+00AC): aparece como marca de continuacion de linea en libros de doble columna; se elimina
- Ligaduras OCR (`fi`, `fl`, `ff`, `st`, etc.): se expanden
- Simbolos de ruido puro (`■`, `•`, `£`, `€`): se eliminan
- Comillas tipograficas: se normalizan a ASCII
- Filtro de densidad de vocales: detecta lineas completamente corruptas


In [ ]:
# Mapa de sustituciones de un solo caracter (construido una vez)
_CHAR_MAP = str.maketrans({
    # Caracteres historicos
    '\u017f': 's',   # long-s  ſ
    '\u017F': 'S',
    '\ua75b': 'r',   # rotunda-r  ꝛ
    '\ua75B': 'R',
    # Ligaduras OCR
    '\ufb01': 'fi',  # ﬁ
    '\ufb02': 'fl',  # ﬂ
    '\ufb00': 'ff',  # ﬀ
    '\ufb03': 'ffi', # ﬃ
    '\ufb04': 'ffl', # ﬄ
    '\ufb05': 'st',  # ﬅ
    '\ufb06': 'st',  # ﬆ
    # Digrafos historicos
    '\u00e6': 'ae',  # æ
    '\u00c6': 'AE',
    '\u0153': 'oe',  # œ
    '\u0152': 'OE',
    # Comillas tipograficas -> ASCII
    '\u2018': "'", '\u2019': "'",
    '\u201c': '"', '\u201d': '"',
    '\u201e': '"', '\u201a': "'",
    '\u00ab': '',   # << comillas angulares (ruido frecuente)
    '\u00bb': '',   # >>
    # Simbolos de ruido
    '\u25a0': ' ',  # cuadrado negro ■
    '\u2022': ' ',  # bala •
    '\u00a3': ' ',  # libra £ (OCR en textos historicos)
    '\u20ac': ' ',  # euro € (imposible en textos historicos; artefacto)
    '\u00ac': '',   # not-sign ¬ (hiphen de columna doble)
    '\u00a7': ' ',  # section sign §
    # Espacios especiales
    '\u00a0': ' ',  # non-breaking space
    '\u200b': '',   # zero-width space
    '\u200c': '',   # zero-width non-joiner
    '\u200d': '',   # zero-width joiner
    '\ufffd': ' ',  # replacement character
    '\u00ad': '',   # soft hyphen
})

_METADATA_RE = re.compile(
    r'(?i)(copyright|google|proquest|digitized\s+by|archive\.org|hathitrust|'
    r'british\s+library|internet\s+archive|university\s+library|'
    r'gallica|europeana|biblioteca\s+nacional|biblioteca\s+digital|'
    r'scanned\s+by|uploaded\s+by)[^\n]*\n?'
)
_URL_RE = re.compile(r'https?://\S+|www\.\S+')
_PAGENUM_RE = re.compile(
    r'[-\u2014\u2013]\s*\d{1,4}\s*[-\u2014\u2013]'
    r'|(?i)\b(p[a\u00e1]g|fol|p)\s*\.\s*\d+'
)
_LONE_NUM_RE = re.compile(r'(?<!\w)\d{1,2}(?!\w)')
_CTRL_RE     = re.compile(r'[\x00-\x08\x0e-\x1f\x7f]')
_ISOL_SYM_RE = re.compile(r'(?<!\w)[#%@\\|~`<>{}\[\]\*\^](?!\w)')
_REPTSYM_RE  = re.compile(r'([^\w\s])\1{2,}')
_WS_RE       = re.compile(r' {2,}')


def clean_text_v4(text: str) -> str:
    '''Limpieza profunda para documentos historicos con ruido OCR.

    Preserva caracteristicas linguisticas utiles para clasificacion por decada
    (variantes ortograficas historicas, vocabulario de epoca) mientras elimina
    artefactos de digitalizacion.
    '''
    if not isinstance(text, str):
        text = str(text)

    # 1. Normalizacion Unicode
    text = unicodedata.normalize('NFC', text)

    # 2. Sustituciones de caracter unico
    text = text.translate(_CHAR_MAP)

    # 3. Reunir palabras partidas por guion al final de linea
    text = re.sub(r'(?<=\w)[\-\u2010\u2011\u2012\u2013\u2014]\s*\n\s*(?=\w)', '', text)
    text = re.sub(r'(?<=\w)[\-]\s+(?=[a-z\u00c0-\u024f])', '', text)

    # 4. Metadatos de bibliotecas digitales
    text = _METADATA_RE.sub(' ', text)
    text = _URL_RE.sub(' ', text)

    # 5. Numeracion de paginas
    text = _PAGENUM_RE.sub(' ', text)

    # 6. Filtrar lineas corruptas
    clean_lines = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        letters = sum(c.isalpha() for c in line)
        total   = len(line)
        puncts  = sum(not c.isalnum() and not c.isspace() for c in line)
        sym_runs = len(re.findall(r'[^\w\s]{3,}', line))

        # Lineas casi sin letras
        if letters < 3 and total > 5:
            continue
        # Lineas dominadas por puntuacion
        if total > 0 and puncts / total > 0.55:
            continue
        # Rachas de simbolos (OCR basura)
        if sym_runs >= 2:
            continue
        # Lineas sin vocales (completamente ilegibles)
        words_in_line = line.split()
        if len(words_in_line) >= 4:
            vowels = sum(1 for c in line.lower() if c in 'aeiou\u00e1\u00e9\u00ed\u00f3\u00fa\u00e0\u00e8\u00ec\u00f2\u00f9')
            if len(line) > 15 and vowels / len(line) < 0.04:
                continue
        clean_lines.append(line)

    text = ' '.join(clean_lines)

    # 7. Caracteres de control
    text = _CTRL_RE.sub(' ', text)

    # 8. Simbolos aislados
    text = _ISOL_SYM_RE.sub(' ', text)

    # 9. Simbolos repetidos
    text = _REPTSYM_RE.sub(r'\1', text)

    # 10. Numeros de 1-2 digitos aislados (ruido OCR)
    text = _LONE_NUM_RE.sub(' ', text)

    # 11. Espacios multiples
    text = _WS_RE.sub(' ', text).strip()

    return text


print('clean_text_v4 definida.')


In [ ]:
print('Limpiando train...')
df_train['text_clean'] = df_train['text'].apply(clean_text_v4)

print('Limpiando eval...')
df_eval['text_clean']  = df_eval['text'].apply(clean_text_v4)

# Eliminar NaN en decade y forzar tipo int
df_train = df_train.dropna(subset=['decade']).reset_index(drop=True)
df_train['decade'] = pd.to_numeric(df_train['decade'], errors='coerce')
df_train = df_train.dropna(subset=['decade']).reset_index(drop=True)
df_train['decade'] = df_train['decade'].astype(int)

# Conservar solo filas con texto suficiente en train; mantener todo eval
mask = df_train['text_clean'].str.len() >= MIN_TEXT_LEN
n_removed = (~mask).sum()
df_train = df_train[mask].reset_index(drop=True)

short_eval = (df_eval['text_clean'].str.len() < MIN_TEXT_LEN).sum()

print(f'Train: {len(df_train)} filas ({n_removed} removidas por texto muy corto)')
print(f'Eval : {len(df_eval)} filas ({short_eval} con texto corto — se conservan)')

# Estadisticas de limpieza
lens_antes = df_train['text'].astype(str).str.len()
lens_despues = df_train['text_clean'].str.len()
reduccion = (lens_antes - lens_despues).mean()
print(f'Longitud media: {lens_antes.mean():.0f} -> {lens_despues.mean():.0f} '
      f'(reduccion media: {reduccion:.0f} chars)')


## 4. Codificacion de labels y split train/val/test


In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(df_train['decade'])
NUM_CLASSES = len(label_encoder.classes_)
print(f'Clases: {NUM_CLASSES} | Decadas: {label_encoder.classes_}')

with open(ppath('label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_encoder, f)

print('LabelEncoder guardado.')


## 5. Augmentacion de datos


In [ ]:
def augment_historical(text: str, p: float = 0.25) -> str:
    '''Variantes ortograficas historicas del espanol (s.XV-XIX).

    Simula la variacion u/v, i/y, s/z propias de la epoca para
    generar ejemplos adicionales sin alterar el contenido semantico.
    '''
    result = []
    for c in text:
        r = random.random()
        if c.islower():
            if c in 'uv' and r < p:
                result.append('v' if c == 'u' else 'u')
            elif c in 'iy' and r < p * 0.5:
                result.append('y' if c == 'i' else 'i')
            else:
                result.append(c)
        else:
            result.append(c)
    return ''.join(result)


def augment_word_dropout(text: str, p: float = 0.1) -> str:
    '''Elimina aleatoriamente palabras para mejorar robustez.'''
    words = text.split()
    return ' '.join(w for w in words if random.random() > p)


df_aug = df_train.copy()

for decade_val in df_train['decade'].unique():
    subset = df_train[df_train['decade'] == decade_val]
    n_need = max(0, MIN_SAMPLES_CLASS - len(subset))
    if n_need > 0:
        extra = subset.sample(n=n_need, replace=True, random_state=SEED).copy()
        # Alternar entre los dos tipos de augmentacion
        extra['text_clean'] = [
            augment_historical(t, 0.25) if i % 2 == 0 else augment_word_dropout(t, 0.08)
            for i, t in enumerate(extra['text_clean'])
        ]
        df_aug = pd.concat([df_aug, extra], ignore_index=True)

df_aug['label'] = label_encoder.transform(df_aug['decade'].astype(int))
print(f'Original: {len(df_train)} | Augmentado: {len(df_aug)}')
print(f'Muestras minimas por clase despues de aug: {df_aug["label"].value_counts().min()}')


In [ ]:
X_all = df_aug['text_clean'].astype(str).to_numpy(dtype=object)
y_all = df_aug['label'].values

assert np.issubdtype(y_all.dtype, np.integer), f'y_all debe ser entero, es {y_all.dtype}'
assert not np.any(pd.isna(y_all)), 'y_all contiene NaN'

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all)
X_val, X_test_int, y_val, y_test_int = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test interno: {len(X_test_int)}')

# Pesos de clase para manejar desbalance residual
cw_arr  = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw_arr)}
print(f'Pesos (min/max): {cw_arr.min():.3f} / {cw_arr.max():.3f}')

# One-hot para label smoothing en redes Keras
y_train_oh    = tf.keras.utils.to_categorical(y_train,    NUM_CLASSES)
y_val_oh      = tf.keras.utils.to_categorical(y_val,      NUM_CLASSES)
y_test_int_oh = tf.keras.utils.to_categorical(y_test_int, NUM_CLASSES)

smooth_loss = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)


## 6. Extraccion de features

Se construyen tres tipos de representacion:
- **TF-IDF de palabras**: captura cambios de vocabulario entre decadas
- **TF-IDF de caracteres**: captura patrones morfologicos y ortograficos del periodo
- **Secuencias de palabras**: para modelos secuenciales (BiLSTM)
- **Secuencias de caracteres**: para CharCNN (robusto a OCR)
- **Features artesanales**: estadisticas del texto que complementan las anteriores


In [ ]:
# --- TF-IDF de palabras ---
print('Construyendo TF-IDF de palabras...')
tfidf_word = TfidfVectorizer(
    max_features=MAX_VOCAB, ngram_range=(1, 3), analyzer='word',
    sublinear_tf=True, min_df=3, max_df=0.95,
    strip_accents=None,
    token_pattern=r'(?u)\b[\w\u00C0-\u024F]{2,}\b',
)
X_train_tw_sp = tfidf_word.fit_transform(X_train)
X_val_tw_sp   = tfidf_word.transform(X_val)
X_test_tw_sp  = tfidf_word.transform(X_test_int)
X_eval_tw_sp  = tfidf_word.transform(df_eval['text_clean'].astype(str))

# Version densa para redes Keras
X_train_tfidf = X_train_tw_sp.toarray().astype(np.float32)
X_val_tfidf   = X_val_tw_sp.toarray().astype(np.float32)
X_test_tfidf  = X_test_tw_sp.toarray().astype(np.float32)
X_eval_tfidf  = X_eval_tw_sp.toarray().astype(np.float32)

with open(ppath('tfidf_word.pkl'), 'wb') as f:
    pickle.dump(tfidf_word, f)
print(f'Word TF-IDF: {X_train_tfidf.shape}')

# --- TF-IDF de caracteres ---
print('Construyendo TF-IDF de caracteres...')
tfidf_char = TfidfVectorizer(
    max_features=50_000, ngram_range=(2, 6), analyzer='char_wb',
    sublinear_tf=True, min_df=3, max_df=0.95,
)
X_train_tc_sp = tfidf_char.fit_transform(X_train)
X_val_tc_sp   = tfidf_char.transform(X_val)
X_test_tc_sp  = tfidf_char.transform(X_test_int)
X_eval_tc_sp  = tfidf_char.transform(df_eval['text_clean'].astype(str))

X_train_ctfidf = X_train_tc_sp.toarray().astype(np.float32)
X_val_ctfidf   = X_val_tc_sp.toarray().astype(np.float32)
X_test_ctfidf  = X_test_tc_sp.toarray().astype(np.float32)
X_eval_ctfidf  = X_eval_tc_sp.toarray().astype(np.float32)

with open(ppath('tfidf_char.pkl'), 'wb') as f:
    pickle.dump(tfidf_char, f)
print(f'Char TF-IDF: {X_train_ctfidf.shape}')

# --- TF-IDF combinado (sparse) para LightGBM ---
print('Combinando para LightGBM (sparse)...')
X_train_comb_sp = sparse_hstack([X_train_tw_sp, X_train_tc_sp]).tocsr()
X_val_comb_sp   = sparse_hstack([X_val_tw_sp,   X_val_tc_sp  ]).tocsr()
X_test_comb_sp  = sparse_hstack([X_test_tw_sp,  X_test_tc_sp ]).tocsr()
X_eval_comb_sp  = sparse_hstack([X_eval_tw_sp,  X_eval_tc_sp ]).tocsr()
print(f'Combinado sparse: {X_train_comb_sp.shape}')


In [ ]:
# --- Secuencias de palabras (para BiLSTM) ---
print('Construyendo secuencias de palabras...')
tokenizer_seq = Tokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
tokenizer_seq.fit_on_texts(X_train)

def to_seq(texts):
    return pad_sequences(
        tokenizer_seq.texts_to_sequences(texts),
        maxlen=MAX_LEN, padding='post', truncating='post'
    )

X_train_seq = to_seq(X_train)
X_val_seq   = to_seq(X_val)
X_test_seq  = to_seq(X_test_int)
X_eval_seq  = to_seq(df_eval['text_clean'].astype(str).values)
VOCAB_SIZE   = min(len(tokenizer_seq.word_index) + 1, MAX_VOCAB)

with open(ppath('tokenizer_seq.pkl'), 'wb') as f:
    pickle.dump(tokenizer_seq, f)
print(f'Secuencias palabras: {X_train_seq.shape} | vocab: {VOCAB_SIZE}')

# --- Secuencias de caracteres (para CharCNN) ---
print('Construyendo secuencias de caracteres...')
CHAR_CHARS = (
    'abcdefghijklmnopqrstuvwxyz'
    '\u00e1\u00e9\u00ed\u00f3\u00fa\u00fc\u00f1'
    '\u00e0\u00e8\u00ec\u00f2\u00f9'
    '\u00e2\u00ea\u00ee\u00f4\u00fb'
    '\u00e4\u00eb\u00ef\u00f6'
    '0123456789 .,;:!?-\'\n'
)
CHAR2IDX  = {c: i + 2 for i, c in enumerate(CHAR_CHARS)}
CHAR_VOCAB = len(CHAR2IDX) + 2


def txt2char(text, ml=CHAR_MAX_LEN):
    s = [CHAR2IDX.get(c.lower(), 1) for c in str(text)[:ml]]
    return s + [0] * max(0, ml - len(s))


X_train_char = np.array([txt2char(t) for t in X_train],              dtype=np.int32)
X_val_char   = np.array([txt2char(t) for t in X_val],                dtype=np.int32)
X_test_char  = np.array([txt2char(t) for t in X_test_int],           dtype=np.int32)
X_eval_char  = np.array([txt2char(t) for t in df_eval['text_clean'].astype(str)], dtype=np.int32)

with open(ppath('char2idx.pkl'), 'wb') as f:
    pickle.dump({'char2idx': CHAR2IDX, 'char_vocab': CHAR_VOCAB, 'char_max_len': CHAR_MAX_LEN}, f)
print(f'Secuencias chars: {X_train_char.shape} | char_vocab: {CHAR_VOCAB}')


In [ ]:
_LATIN_WORDS = frozenset([
    'et', 'est', 'in', 'de', 'ad', 'per', 'cum', 'ut', 'non', 'sed',
    'quod', 'qui', 'quae', 'que', 'vel', 'ex', 'pro', 'ac', 'atque',
    'autem', 'enim', 'nam', 'nec', 'neque', 'quia', 'si', 'sub', 'super',
])
_VOWELS_SET = frozenset('aeiou\u00e1\u00e9\u00ed\u00f3\u00fa')


def extract_handcrafted(texts: list) -> np.ndarray:
    '''Features estadisticas del texto que capturan patrones temporales.

    Incluye: longitud, diversidad lexica, ratio de palabras latinas,
    densidad de vocales, longitud media de palabra, y frecuencias de
    caracteres especificos del espanol historico.
    '''
    feats = []
    for text in texts:
        text = str(text)
        words = text.split()
        n_words = max(len(words), 1)
        n_chars = max(len(text), 1)

        # Longitud
        f_len_log    = np.log1p(n_chars)
        f_words_log  = np.log1p(n_words)

        # Diversidad lexica (type-token ratio con suavizado)
        unique_words  = len(set(w.lower() for w in words))
        f_ttr         = unique_words / n_words

        # Longitud media de palabra
        f_avg_wlen    = np.mean([len(w) for w in words]) if words else 0.0

        # Ratio de palabras latinas (indicador de periodo)
        latin_cnt     = sum(1 for w in words if w.lower() in _LATIN_WORDS)
        f_latin_ratio = latin_cnt / n_words

        # Densidad de vocales
        vowels        = sum(1 for c in text.lower() if c in _VOWELS_SET)
        f_vowel_dens  = vowels / n_chars

        # Frecuencia de caracteres historicos especificos del espanol
        f_ny    = text.lower().count('\u00f1') / n_chars  # n-tilde
        f_uv    = (text.lower().count('v') + text.lower().count('u')) / max(n_chars, 1)
        f_puncts = sum(not c.isalnum() and not c.isspace() for c in text) / n_chars

        # Longitud de oraciones estimada (por puntos y punto-y-coma)
        sents = max(text.count('.') + text.count(';') + text.count('?') + text.count('!'), 1)
        f_sent_len = n_words / sents

        # Ratio de mayusculas
        f_cap_ratio = sum(c.isupper() for c in text) / max(sum(c.isalpha() for c in text), 1)

        feats.append([
            f_len_log, f_words_log, f_ttr, f_avg_wlen,
            f_latin_ratio, f_vowel_dens, f_ny, f_uv,
            f_puncts, f_sent_len, f_cap_ratio,
        ])
    return np.array(feats, dtype=np.float32)


print('Extrayendo features artesanales...')
X_train_hc = extract_handcrafted(X_train)
X_val_hc   = extract_handcrafted(X_val)
X_test_hc  = extract_handcrafted(X_test_int)
X_eval_hc  = extract_handcrafted(df_eval['text_clean'].astype(str).tolist())
print(f'Features artesanales: {X_train_hc.shape}')


## 7. Callbacks y utilidades comunes

- **ManualStopCallback**: Para detener cualquier modelo ejecuta `MANUAL_STOP['stop'] = True` en otra celda
- **EarlyStopping**: Para automaticamente si `val_accuracy` no mejora en `patience` epocas


In [ ]:
MANUAL_STOP = {'stop': False}


class ManualStopCallback(Callback):
    '''Detiene el entrenamiento cuando MANUAL_STOP["stop"] == True.

    Para detener un modelo en ejecucion, corre en otra celda:
        MANUAL_STOP["stop"] = True
    '''
    def on_epoch_end(self, epoch, logs=None):
        if MANUAL_STOP.get('stop', False):
            print(f'\n[PARADA MANUAL] Detenido en epoca {epoch + 1}.')
            self.model.stop_training = True
            MANUAL_STOP['stop'] = False


def get_callbacks(model_name, monitor='val_accuracy', patience=15):
    '''Conjunto estandar de callbacks para todos los modelos Keras.'''
    return [
        EarlyStopping(
            monitor=monitor, patience=patience,
            restore_best_weights=True, verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=6,
            min_lr=1e-7, verbose=1,
        ),
        ModelCheckpoint(
            kpath(f'best_{model_name}.keras'),
            monitor=monitor, save_best_only=True, verbose=0,
        ),
        ManualStopCallback(),
    ]


# Diccionario global de resultados — se rellena a medida que se entrenan modelos
MODELS_INFO = {}

print('Callbacks definidos.')
print('Para detener un modelo: ejecuta   MANUAL_STOP["stop"] = True   en otra celda.')


## 8. Modelo 1 — MLP Combinado

**Mejor modelo de los Intentos anteriores, mejorado.**

El MLP en TF-IDF fue el modelo con mejor desempeno en los Intentos 1-3 (submission `submission_mlp.csv`).

Mejoras en Intento 4:
- Dos ramas independientes: una para word TF-IDF, otra para char TF-IDF
- Features artesanales concatenados antes del head final
- Regularizacion L2 mas fuerte en capas iniciales
- Label smoothing 0.1 y pesos de clase


In [ ]:
def build_mlp_combined(word_dim, char_dim, hc_dim, num_classes):
    '''MLP de dos ramas sobre TF-IDF de palabras y caracteres.

    Rama word: captura vocabulario y n-gramas de palabras del periodo.
    Rama char: captura morfologia y patrones ortograficos historicos.
    Se fusionan con las features artesanales antes de la capa de clasificacion.
    '''
    # Rama word TF-IDF
    inp_word = keras.Input(shape=(word_dim,), name='word_tfidf')
    w = layers.Dense(1024, kernel_regularizer=keras.regularizers.l2(1e-4))(inp_word)
    w = layers.BatchNormalization()(w)
    w = layers.Activation('relu')(w)
    w = layers.Dropout(0.5)(w)
    w = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(1e-4))(w)
    w = layers.BatchNormalization()(w)
    w = layers.Activation('relu')(w)
    w = layers.Dropout(0.4)(w)

    # Rama char TF-IDF
    inp_char = keras.Input(shape=(char_dim,), name='char_tfidf')
    c = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(1e-4))(inp_char)
    c = layers.BatchNormalization()(c)
    c = layers.Activation('relu')(c)
    c = layers.Dropout(0.5)(c)
    c = layers.Dense(256, kernel_regularizer=keras.regularizers.l2(1e-4))(c)
    c = layers.BatchNormalization()(c)
    c = layers.Activation('relu')(c)
    c = layers.Dropout(0.4)(c)

    # Rama features artesanales
    inp_hc = keras.Input(shape=(hc_dim,), name='handcrafted')
    h = layers.Dense(64, activation='relu')(inp_hc)
    h = layers.Dropout(0.3)(h)

    # Fusion
    merged = layers.Concatenate()([w, c, h])
    x = layers.Dense(512, kernel_regularizer=keras.regularizers.l2(5e-5))(merged)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs=[inp_word, inp_char, inp_hc], outputs=out, name='MLP_Combinado')


if TRAIN_MLP:
    with TF_STRATEGY.scope():
        model_mlp = build_mlp_combined(
            X_train_tfidf.shape[1],
            X_train_ctfidf.shape[1],
            X_train_hc.shape[1],
            NUM_CLASSES,
        )
        model_mlp.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss=smooth_loss,
            metrics=['accuracy'],
        )
    model_mlp.summary()
else:
    print('TRAIN_MLP=False. Saltando arquitectura.')


In [ ]:
if TRAIN_MLP:
    history_mlp = model_mlp.fit(
        [X_train_tfidf, X_train_ctfidf, X_train_hc], y_train_oh,
        validation_data=([X_val_tfidf, X_val_ctfidf, X_val_hc], y_val_oh),
        epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        class_weight=cw_dict,
        callbacks=get_callbacks('mlp_combinado'),
        verbose=1,
    )
    model_mlp.save(kpath('modelo_mlp_combinado_final.keras'))

    mlp_val_probs  = model_mlp.predict([X_val_tfidf,      X_val_ctfidf,   X_val_hc],   verbose=0)
    mlp_test_probs = model_mlp.predict([X_test_tfidf,     X_test_ctfidf,  X_test_hc],  verbose=0)
    mlp_eval_probs = model_mlp.predict([X_eval_tfidf,     X_eval_ctfidf,  X_eval_hc],  verbose=0)

    mlp_val_acc  = accuracy_score(y_val,      np.argmax(mlp_val_probs,  1))
    mlp_test_acc = accuracy_score(y_test_int, np.argmax(mlp_test_probs, 1))
    print(f'MLP Combinado — Val: {mlp_val_acc:.4f} | Test interno: {mlp_test_acc:.4f}')

    MODELS_INFO['MLP_Combinado'] = {
        'val': mlp_val_acc, 'test': mlp_test_acc,
        'val_probs':  mlp_val_probs,
        'test_probs': mlp_test_probs,
        'eval_probs': mlp_eval_probs,
    }
else:
    print('TRAIN_MLP=False. Saltando entrenamiento.')


## 9. Modelo 2 — BiLSTM v3

Red LSTM bidireccional con atencion multi-cabeza.

Cambios respecto a BiLSTM v2 (Intento 3):
- Atencion multi-cabeza (`MultiHeadAttention` de Keras) en lugar de Bahdanau; captura dependencias globales
- Mayor numero de unidades en la primera capa LSTM (512 vs 256)
- Conexion residual en la atencion


In [ ]:
def build_bilstm_v3(vocab_size, embed_dim, max_len, num_classes):
    '''BiLSTM con multi-head attention y conexion residual.'''
    inputs = keras.Input(shape=(max_len,))
    x = layers.Embedding(vocab_size, embed_dim, mask_zero=False)(inputs)
    x = layers.SpatialDropout1D(0.3)(x)

    x = layers.Bidirectional(layers.LSTM(
        256, return_sequences=True, dropout=0.2, recurrent_dropout=0.1
    ))(x)
    x = layers.Bidirectional(layers.LSTM(
        128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1
    ))(x)

    # Multi-head attention sobre las salidas de la LSTM
    attn_out = layers.MultiHeadAttention(
        num_heads=4, key_dim=64, dropout=0.1
    )(x, x)
    x = layers.LayerNormalization()(x + attn_out)   # conexion residual

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, out, name='BiLSTM_v3')


if TRAIN_BILSTM:
    with TF_STRATEGY.scope():
        model_bilstm = build_bilstm_v3(VOCAB_SIZE, EMBED_DIM, MAX_LEN, NUM_CLASSES)
        model_bilstm.compile(
            optimizer=keras.optimizers.Adam(5e-4, clipnorm=1.0),
            loss=smooth_loss,
            metrics=['accuracy'],
        )
    model_bilstm.summary()
else:
    print('TRAIN_BILSTM=False. Saltando arquitectura.')


In [ ]:
if TRAIN_BILSTM:
    history_bilstm = model_bilstm.fit(
        X_train_seq, y_train_oh,
        validation_data=(X_val_seq, y_val_oh),
        epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        class_weight=cw_dict,
        callbacks=get_callbacks('bilstm_v3'),
        verbose=1,
    )
    model_bilstm.save(kpath('modelo_bilstm_v3_final.keras'))

    bilstm_val_probs  = model_bilstm.predict(X_val_seq,      verbose=0)
    bilstm_test_probs = model_bilstm.predict(X_test_seq,     verbose=0)
    bilstm_eval_probs = model_bilstm.predict(X_eval_seq,     verbose=0)

    bilstm_val_acc  = accuracy_score(y_val,      np.argmax(bilstm_val_probs,  1))
    bilstm_test_acc = accuracy_score(y_test_int, np.argmax(bilstm_test_probs, 1))
    print(f'BiLSTM v3 — Val: {bilstm_val_acc:.4f} | Test interno: {bilstm_test_acc:.4f}')

    MODELS_INFO['BiLSTM_v3'] = {
        'val': bilstm_val_acc, 'test': bilstm_test_acc,
        'val_probs':  bilstm_val_probs,
        'test_probs': bilstm_test_probs,
        'eval_probs': bilstm_eval_probs,
    }
else:
    print('TRAIN_BILSTM=False. Saltando entrenamiento.')


## 10. Modelo 3 — CharCNN v2

CNN sobre secuencias de caracteres con bloques residuales.

Esencial para este dataset por dos razones:
1. El OCR corrupto (especialmente decadas 150-159) rompe la segmentacion en palabras;
   los patrones de caracteres siguen siendo informativos.
2. Las variantes ortograficas historicas (u/v, i/j, long-s) son señales clave a nivel caracter.

Cambios respecto a CharCNN v1 (Intento 3):
- Bloque residual por cada kernel (suma de entrada + salida del Conv)
- Mayor numero de filtros (512 vs 256)


In [ ]:
def residual_char_block(x, filters, kernel_size):
    '''Bloque residual convolucional para secuencias de caracteres.

    Aplica Conv1D + MaxPool y suma la entrada (proyectada si es necesario)
    para mejorar el flujo de gradiente en redes profundas.
    '''
    shortcut = x
    out = layers.Conv1D(
        filters, kernel_size, activation='relu', padding='same',
        kernel_regularizer=keras.regularizers.l2(5e-5)
    )(x)
    out = layers.Conv1D(filters, kernel_size, activation='relu', padding='same')(out)
    # Proyectar shortcut si el numero de canales cambia
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)
    return layers.Add()([out, shortcut])


def build_charcnn_v2(char_vocab_size, embed_dim, max_len, num_classes):
    '''CharCNN con bloques residuales y kernels de distintos tamanios.

    Cada rama usa un kernel distinto para capturar patrones de
    longitud diferente (morfemas, palabras, frases cortas).
    '''
    inputs = keras.Input(shape=(max_len,), name='char_input')
    emb = layers.Embedding(char_vocab_size, embed_dim)(inputs)
    emb = layers.SpatialDropout1D(0.2)(emb)

    branches = []
    for ks in [3, 5, 7, 11]:
        b = residual_char_block(emb, 512, ks)
        b = layers.MaxPooling1D(3, padding='same')(b)
        b = residual_char_block(b, 256, ks)
        b = layers.GlobalMaxPooling1D()(b)
        branches.append(b)

    x = layers.Concatenate()(branches)   # 4 x 256 = 1024
    x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, out, name='CharCNN_v2')


if TRAIN_CHARCNN:
    with TF_STRATEGY.scope():
        model_charcnn = build_charcnn_v2(CHAR_VOCAB, CHAR_EMBED_DIM, CHAR_MAX_LEN, NUM_CLASSES)
        model_charcnn.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss=smooth_loss,
            metrics=['accuracy'],
        )
    model_charcnn.summary()
else:
    print('TRAIN_CHARCNN=False. Saltando arquitectura.')


In [ ]:
if TRAIN_CHARCNN:
    history_charcnn = model_charcnn.fit(
        X_train_char, y_train_oh,
        validation_data=(X_val_char, y_val_oh),
        epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
        class_weight=cw_dict,
        callbacks=get_callbacks('charcnn_v2'),
        verbose=1,
    )
    model_charcnn.save(kpath('modelo_charcnn_v2_final.keras'))

    charcnn_val_probs  = model_charcnn.predict(X_val_char,   verbose=0)
    charcnn_test_probs = model_charcnn.predict(X_test_char,  verbose=0)
    charcnn_eval_probs = model_charcnn.predict(X_eval_char,  verbose=0)

    charcnn_val_acc  = accuracy_score(y_val,      np.argmax(charcnn_val_probs,  1))
    charcnn_test_acc = accuracy_score(y_test_int, np.argmax(charcnn_test_probs, 1))
    print(f'CharCNN v2 — Val: {charcnn_val_acc:.4f} | Test interno: {charcnn_test_acc:.4f}')

    MODELS_INFO['CharCNN_v2'] = {
        'val': charcnn_val_acc, 'test': charcnn_test_acc,
        'val_probs':  charcnn_val_probs,
        'test_probs': charcnn_test_probs,
        'eval_probs': charcnn_eval_probs,
    }
else:
    print('TRAIN_CHARCNN=False. Saltando entrenamiento.')


## 11. Modelo 4 — LightGBM (nuevo)

Gradient boosting sobre la matriz TF-IDF combinada (word + char) en formato **sparse** — sin
conversion a denso, por lo que usa poca memoria.

Por que LightGBM para clasificacion de texto:
- Utiliza todos los nucleos CPU con `n_jobs=-1`
- No requiere GPU para ser competitivo
- Sobre features TF-IDF es frecuentemente el metodo con mejor desempeno absoluto
- El entrenamiento es rapido comparado con redes neurales
- `predict_proba` devuelve distribuciones de probabilidad compatibles con el ensemble


In [ ]:
if TRAIN_LGBM:
    print('Entrenando LightGBM...')
    lgbm_model = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=NUM_CLASSES,
        metric='multi_logloss',
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=127,
        max_depth=10,
        min_child_samples=15,
        feature_fraction=0.6,
        bagging_fraction=0.8,
        bagging_freq=5,
        lambda_l1=0.1,
        lambda_l2=0.2,
        n_jobs=-1,           # usa todos los nucleos CPU disponibles
        random_state=SEED,
        verbose=-1,
    )

    # Callbacks para LightGBM: early stopping y logging
    lgbm_callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50),
    ]

    lgbm_model.fit(
        X_train_comb_sp, y_train,
        eval_set=[(X_val_comb_sp, y_val)],
        callbacks=lgbm_callbacks,
    )

    # Guardar modelo
    with open(ppath('lgbm_model.pkl'), 'wb') as f:
        pickle.dump(lgbm_model, f)
    lgbm_model.booster_.save_model(lpath('lgbm_model.txt'))

    lgbm_val_probs  = lgbm_model.predict_proba(X_val_comb_sp).astype(np.float32)
    lgbm_test_probs = lgbm_model.predict_proba(X_test_comb_sp).astype(np.float32)
    lgbm_eval_probs = lgbm_model.predict_proba(X_eval_comb_sp).astype(np.float32)

    lgbm_val_acc  = accuracy_score(y_val,      np.argmax(lgbm_val_probs,  1))
    lgbm_test_acc = accuracy_score(y_test_int, np.argmax(lgbm_test_probs, 1))
    print(f'LightGBM — Val: {lgbm_val_acc:.4f} | Test interno: {lgbm_test_acc:.4f}')

    MODELS_INFO['LightGBM'] = {
        'val': lgbm_val_acc, 'test': lgbm_test_acc,
        'val_probs':  lgbm_val_probs,
        'test_probs': lgbm_test_probs,
        'eval_probs': lgbm_eval_probs,
    }
else:
    print('TRAIN_LGBM=False. Saltando entrenamiento.')


## 12. Modelo 5 — BERT Espanol / XLM-RoBERTa (opcional)

Fine-tuning de modelo preentrenado para clasificacion de secuencias.

Se usa `xlm-roberta-base` en lugar de `bert-base-spanish` del Intento 3 porque:
- Fue entrenado en texto de 100 idiomas incluyendo espanol, latin e italiano
- Su tokenizacion BPE es mas robusta a OCR corrupto
- Mejores resultados en benchmarks de espanol historico

**Recomendacion**: activar `TRAIN_BERT = True` solo si hay GPU disponible.
En CPU el entrenamiento puede tardar varios dias.


In [ ]:
if not TRAIN_BERT:
    print('TRAIN_BERT=False. Para activar cambia la variable en la celda de config.')
elif not BERT_AVAILABLE:
    print('PyTorch/Transformers no instalados. BERT no disponible.')
else:
    BERT_NAME   = 'xlm-roberta-base'
    BERT_MAXLEN = 512
    BERT_BATCH  = 16
    BERT_EPOCHS = 5
    BERT_LR     = 2e-5
    BERT_OUTDIR = PROCESS / 'bert' / 'best_model'

    class HistoricalDataset(Dataset):
        '''Dataset PyTorch para textos historicos con tokenizacion BERT.'''
        def __init__(self, texts, labels, tokenizer):
            self.texts     = texts
            self.labels    = labels
            self.tokenizer = tokenizer

        def __len__(self): return len(self.texts)

        def __getitem__(self, idx):
            enc = self.tokenizer(
                str(self.texts[idx]),
                max_length=BERT_MAXLEN, padding='max_length',
                truncation=True, return_tensors='pt',
            )
            return {
                'input_ids':      enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
            }

    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        BERT_NAME, num_labels=NUM_CLASSES, ignore_mismatched_sizes=True
    ).to(DEVICE_PT)

    # Congelar las primeras 8 capas del encoder para fine-tuning eficiente
    for name, param in bert_model.named_parameters():
        if any(f'layer.{i}.' in name for i in range(8)):
            param.requires_grad = False

    trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in bert_model.parameters())
    print(f'BERT ({BERT_NAME}): {trainable:,} params entrenables / {total:,} totales')

    ds_train_bert = HistoricalDataset(X_train, y_train, bert_tokenizer)
    ds_val_bert   = HistoricalDataset(X_val,   y_val,   bert_tokenizer)
    dl_train_bert = DataLoader(ds_train_bert, batch_size=BERT_BATCH, shuffle=True, num_workers=0)
    dl_val_bert   = DataLoader(ds_val_bert,   batch_size=BERT_BATCH,               num_workers=0)

    no_decay = ['bias', 'LayerNorm.weight']
    bert_params = [
        {'params': [p for n, p in bert_model.named_parameters()
                    if p.requires_grad and not any(nd in n for nd in no_decay)],
         'weight_decay': 0.01},
        {'params': [p for n, p in bert_model.named_parameters()
                    if p.requires_grad and any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    bert_optimizer = torch.optim.AdamW(bert_params, lr=BERT_LR)
    total_steps    = len(dl_train_bert) * BERT_EPOCHS
    bert_scheduler = get_linear_schedule_with_warmup(
        bert_optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    print('Setup BERT listo.')


In [ ]:
if TRAIN_BERT and BERT_AVAILABLE:
    best_bert_val = 0.0

    for epoch in range(BERT_EPOCHS):
        bert_model.train()
        total_loss = 0.0
        for batch in dl_train_bert:
            bert_optimizer.zero_grad()
            output = bert_model(
                input_ids=batch['input_ids'].to(DEVICE_PT),
                attention_mask=batch['attention_mask'].to(DEVICE_PT),
                labels=batch['labels'].to(DEVICE_PT),
            )
            output.loss.backward()
            torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
            bert_optimizer.step()
            bert_scheduler.step()
            total_loss += output.loss.item()

        bert_model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in dl_val_bert:
                logits = bert_model(
                    input_ids=batch['input_ids'].to(DEVICE_PT),
                    attention_mask=batch['attention_mask'].to(DEVICE_PT),
                ).logits
                all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                all_labels.extend(batch['labels'].numpy())

        val_acc = accuracy_score(all_labels, all_preds)
        avg_loss = total_loss / len(dl_train_bert)
        print(f'BERT Epoch {epoch + 1}/{BERT_EPOCHS} | Loss: {avg_loss:.4f} | Val: {val_acc:.4f}')

        if val_acc > best_bert_val:
            best_bert_val = val_acc
            bert_model.save_pretrained(str(BERT_OUTDIR))
            bert_tokenizer.save_pretrained(str(BERT_OUTDIR))
            print(f'  Mejor BERT guardado (val={best_bert_val:.4f})')

        if MANUAL_STOP.get('stop', False):
            print('[PARADA MANUAL] BERT detenido.')
            MANUAL_STOP['stop'] = False
            break

    # Obtener probabilidades para el ensemble
    def get_bert_probs(texts, tokenizer, model, batch_size=BERT_BATCH):
        '''Obtiene probabilidades softmax del modelo BERT sobre una lista de textos.'''
        model.eval()
        probs_list = []
        for i in range(0, len(texts), batch_size):
            batch_texts = [str(t) for t in texts[i:i + batch_size]]
            enc = tokenizer(
                batch_texts, max_length=BERT_MAXLEN,
                padding=True, truncation=True, return_tensors='pt',
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc['input_ids'].to(DEVICE_PT),
                    attention_mask=enc['attention_mask'].to(DEVICE_PT),
                ).logits
            probs_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.vstack(probs_list).astype(np.float32)

    bert_val_probs  = get_bert_probs(X_val,      bert_tokenizer, bert_model)
    bert_test_probs = get_bert_probs(X_test_int, bert_tokenizer, bert_model)
    bert_eval_probs = get_bert_probs(
        df_eval['text_clean'].astype(str).values, bert_tokenizer, bert_model)

    bert_test_acc = accuracy_score(y_test_int, np.argmax(bert_test_probs, 1))
    print(f'BERT (XLM-R) — Val: {best_bert_val:.4f} | Test interno: {bert_test_acc:.4f}')

    MODELS_INFO['BERT_XLM'] = {
        'val': best_bert_val, 'test': bert_test_acc,
        'val_probs':  bert_val_probs,
        'test_probs': bert_test_probs,
        'eval_probs': bert_eval_probs,
    }


## 13. Ensemble ponderado y comparacion de modelos

Se combina la salida de todos los modelos entrenados mediante un promedio
ponderado por su `val_accuracy`. Solo se incluyen modelos que superen
`ENSEMBLE_THRESHOLD`.


In [ ]:
if not MODELS_INFO:
    print('No hay modelos entrenados. Entrenar al menos uno antes de correr el ensemble.')
else:
    # Filtrar por umbral minimo
    included = {n: d for n, d in MODELS_INFO.items() if d['val'] >= ENSEMBLE_THRESHOLD}
    if not included:
        included = MODELS_INFO  # fallback: incluir todos

    total_w = sum(d['val'] for d in included.values())
    weights = {n: d['val'] / total_w for n, d in included.items()}

    print('Modelos en ensemble:')
    for name, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f'  {name:<20} val={MODELS_INFO[name]["val"]:.4f}  peso={w:.3f}')

    ens_val_probs  = sum(w * included[n]['val_probs']  for n, w in weights.items())
    ens_test_probs = sum(w * included[n]['test_probs'] for n, w in weights.items())
    ens_eval_probs = sum(w * included[n]['eval_probs'] for n, w in weights.items())

    ens_val_acc  = accuracy_score(y_val,      np.argmax(ens_val_probs,  1))
    ens_test_acc = accuracy_score(y_test_int, np.argmax(ens_test_probs, 1))
    print(f'\nENSAMBLE — Val: {ens_val_acc:.4f} | Test interno: {ens_test_acc:.4f}')

    MODELS_INFO['ENSEMBLE'] = {
        'val': ens_val_acc, 'test': ens_test_acc,
        'eval_probs': ens_eval_probs,
    }


In [ ]:
if MODELS_INFO:
    df_res = pd.DataFrame({
        name: {'val_acc': d['val'], 'test_acc': d['test']}
        for name, d in MODELS_INFO.items()
    }).T.sort_values('val_acc', ascending=False)

    print('Resumen de modelos:')
    print(df_res.to_string(float_format='{:.4f}'.format))

    fig, ax = plt.subplots(figsize=(max(8, len(df_res) * 1.5), 5))
    x = range(len(df_res))
    ax.bar([i - 0.2 for i in x], df_res['val_acc'],  width=0.4, label='Val',          color='steelblue', alpha=0.8)
    ax.bar([i + 0.2 for i in x], df_res['test_acc'], width=0.4, label='Test interno', color='coral',     alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_res.index, rotation=30, ha='right')
    ax.set_ylabel('Accuracy')
    ax.set_title('Comparacion de modelos — Intento 4')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ipath('comparacion_intento4.png'), dpi=100, bbox_inches='tight')
    plt.show()

    best_name = df_res.index[0]
    print(f'Mejor modelo: {best_name} (val={df_res.loc[best_name, "val_acc"]:.4f})')


## 14. Generacion de submission para Kaggle


In [ ]:
if 'ENSEMBLE' in MODELS_INFO:
    # Usar el ensemble si esta disponible
    eval_probs_final = MODELS_INFO['ENSEMBLE']['eval_probs']
    source_model     = 'ensemble'
elif MODELS_INFO:
    # Usar el mejor modelo individual
    best_name        = max(MODELS_INFO, key=lambda n: MODELS_INFO[n]['val'])
    eval_probs_final = MODELS_INFO[best_name]['eval_probs']
    source_model     = best_name
    print(f'Ensemble no disponible. Usando mejor modelo individual: {best_name}')
else:
    raise RuntimeError('No hay modelos entrenados. No se puede generar submission.')

kaggle_preds = label_encoder.inverse_transform(np.argmax(eval_probs_final, axis=1))

submission   = pd.DataFrame({'id': df_eval['id'], 'answer': kaggle_preds})
sub_path     = f'submission_intento4_{source_model}.csv'
submission.to_csv(sub_path, index=False)

print(f'Submission guardado en: {sub_path}')
print(f'Filas: {len(submission)}')
print(f'Decadas predichas (muestra): {sorted(submission["answer"].unique())[:10]}')
print(submission.head())


## 15. Guardado de artefactos


In [ ]:
# Metricas finales en JSON
metrics_out = {
    'intento': 4,
    'num_classes': int(NUM_CLASSES),
    'train_size': int(len(X_train)),
    'val_size': int(len(X_val)),
    'modelos': {},
}
for name, d in MODELS_INFO.items():
    metrics_out['modelos'][name] = {
        'val_acc': float(d['val']),
        'test_acc': float(d['test']),
    }

metrics_path = str(PROCESS / 'metrics_intento4.json')
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_out, f, indent=2, ensure_ascii=False)
print(f'Metricas guardadas en: {metrics_path}')

# Mover submission a carpeta entregas (si existe)
entregas_dir = Path('./entregas')
if entregas_dir.exists() and 'sub_path' in dir():
    import shutil
    dest = entregas_dir / sub_path
    shutil.copy(sub_path, dest)
    print(f'Submission copiado a: {dest}')

print('Artefactos guardados:')
for p in sorted(PROCESS.rglob('*')):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        print(f'  {p.relative_to(PROCESS)}  ({size_kb:.1f} KB)')
